In [2]:
# ==== Model Builder (Option A: intersection join, t+1 target) ====
# Dependencies
import warnings, math, os
from pathlib import Path
import pandas as pd, numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import joblib

# ---- paths ----

PROC = Path("./processed") 
OUT  = Path("./models")

warnings.filterwarnings("ignore")

# ---- helpers ----
META_COLS = {"quarter_end","year","quarter","quarter_label"}

def _pick_value_col(df: pd.DataFrame) -> str:
    """
    From a standardized processed CSV (with metadata columns present),
    return the single *numeric* series column to use as the value.
    """
    candidates = [c for c in df.columns if c not in META_COLS]
    # Prefer the one that is most numeric
    best, best_non_na = None, -1
    for c in candidates:
        num = pd.to_numeric(df[c], errors="coerce")
        non_na = int(num.notna().sum())
        if non_na > best_non_na:
            best, best_non_na = c, non_na
    if best is None:
        raise ValueError("Could not find a numeric value column.")
    return best

def load_series(proc_filename: str, rename_to: str) -> pd.Series:
    """Load a processed CSV and return a numeric Series indexed by quarter_end."""
    df = pd.read_csv(PROC / proc_filename, parse_dates=["quarter_end"])
    vcol = _pick_value_col(df)
    ser = (pd.to_numeric(df[vcol], errors="coerce")
             .rename(rename_to)
             .set_axis(df["quarter_end"]))
    # If duplicates exist per quarter_end (shouldn't), take last
    ser = ser.groupby(ser.index).last().sort_index()
    return ser

def rmspe(y, yhat):
    mask = y != 0
    if mask.any():
        return math.sqrt(np.mean(((yhat[mask] - y[mask]) / y[mask])**2)) * 100
    return np.nan

def print_metrics(y_true, y_pred, label=""):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_pred - y_true) / np.where(y_true==0, np.nan, y_true))) * 100
    print(f"{label}RMSE={rmse:6.3f} | MAE={mae:6.3f} | R2={r2:6.3f} | MAPE={mape:6.2f}%")

# ---- load all series ----
series_map = {
    "quarterly_excess_ret.csv" : "excess_ret",
    "quarterly_cpi_yoy.csv"    : "cpi_yoy",
    "quarterly_gdp_yoy.csv"    : "gdp_yoy",
    "quarterly_repo_level.csv" : "repo",
    "quarterly_repo_chg_bps.csv":"repo_chg_bps",
    "quarterly_rain_anom.csv"  : "rain_anom",
}

loaded = {}
missing_files = []
for fn, name in series_map.items():
    path = PROC / fn
    if not path.exists():
        missing_files.append(fn)
        continue
    loaded[name] = load_series(fn, name)

if missing_files:
    print("⚠ Missing files (skipped):", missing_files)

if not loaded:
    raise RuntimeError("No series loaded. Check PROC path & filenames.")

# ---- inner-join on common quarters ----
df = pd.concat(loaded.values(), axis=1, join="inner").sort_index()
df.index.name = "quarter_end"

# (Optional) quick QA
print("Data coverage:", df.index.min().date(), "→", df.index.max().date(), "| rows:", df.shape[0])
print(df.tail(3))

# ---- target: next-quarter excess return ----
# We predict T+1 excess_ret using features at time T (no leakage).
df["target_excess_ret_next"] = df["excess_ret"].shift(-1)

# ---- simple feature engineering knobs (easy to tweak) ----
# You can add modest lags to economic features to reflect reporting delays if you want.
LAGS = {
    # 'cpi_yoy': 1,
    # 'gdp_yoy': 1,
    # 'repo': 0,
    # 'repo_chg_bps': 0,
    # 'rain_anom': 0,
}
for col, lag in LAGS.items():
    if col in df.columns and lag:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

# Quarter-of-year encodings (cyclical)
per = df.index.to_period("Q")
q_num = per.quarter
df["q_sin"] = np.sin(2*np.pi*(q_num/4.0))
df["q_cos"] = np.cos(2*np.pi*(q_num/4.0))

# Momentum of the spread (helps if there’s persistence)
df["excess_ret_lag1"] = df["excess_ret"].shift(1)
df["excess_ret_ma4"]  = df["excess_ret"].rolling(4).mean()

# ---- finalize features/labels ----
feature_cols = [c for c in df.columns if c not in {"target_excess_ret_next"}]
feature_cols = [c for c in feature_cols if not c.startswith("excess_ret")] + ["excess_ret_lag1","excess_ret_ma4","q_sin","q_cos"]
feature_cols = [c for c in feature_cols if c in df.columns]
# After you build `feature_cols`:
feature_cols = list(dict.fromkeys(feature_cols))  # keep order, drop dups

# If you use repo_chg_bps, drop raw 'repo' level to reduce redundancy:
if "repo" in feature_cols and "repo_chg_bps" in feature_cols:
    feature_cols.remove("repo")

# Re-create X,y cleanly
X = df[feature_cols]
y = df["target_excess_ret_next"]
data = pd.concat([X, y], axis=1).dropna()
X = data[feature_cols]
y = data["target_excess_ret_next"]

print("Final features:", feature_cols)
print("Rows after cleanup:", len(X))

# ---- time-series CV setup ----
# Use expanding-window folds; last fold acts like a recent out-of-sample.
n_splits = 5 if len(data) >= 40 else max(3, len(data)//10)  # heuristic to avoid tiny folds
tscv = TimeSeriesSplit(n_splits=n_splits)

# ---- candidate models ----
candidates = {
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=3.0, random_state=42)),
    ]),
    "lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.01, random_state=42, max_iter=5000)),
    ]),
    "gbr": Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", GradientBoostingRegressor(random_state=42,
                                            n_estimators=600,
                                            learning_rate=0.03,
                                            max_depth=3,
                                            subsample=0.9)),
    ]),
    "rf": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42,
                                        n_estimators=500,
                                        max_depth=None,
                                        min_samples_leaf=2))
    ]),
}

# ---- backtest & model selection ----
cv_rows = []
scores = []
for name, pipe in candidates.items():
    y_pred_full = pd.Series(index=y.index, dtype=float)
    fold_no = 0
    for train_idx, test_idx in tscv.split(X):
        fold_no += 1
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        y_hat = pd.Series(pipe.predict(X_te), index=y_te.index)

        # store per-fold predictions for later inspection
        y_pred_full.loc[y_te.index] = y_hat

        # metrics
        rmse = mean_squared_error(y_te, y_hat, squared=False)
        mae  = mean_absolute_error(y_te, y_hat)
        r2   = r2_score(y_te, y_hat)
        cv_rows.append({
            "model": name, "fold": fold_no, "start": X_te.index.min(), "end": X_te.index.max(),
            "rmse": rmse, "mae": mae, "r2": r2
        })

    # summarize per model
    valid_mask = y_pred_full.notna()
    rmse = mean_squared_error(y[valid_mask], y_pred_full[valid_mask], squared=False)
    mae  = mean_absolute_error(y[valid_mask], y_pred_full[valid_mask])
    r2   = r2_score(y[valid_mask], y_pred_full[valid_mask])
    scores.append((name, rmse, mae, r2))
    print_metrics(y[valid_mask], y_pred_full[valid_mask], label=f"[{name}] ")

cv_df = pd.DataFrame(cv_rows).sort_values(["model","fold"])
cv_df.to_csv(OUT/"cv_folds_metrics.csv", index=False)

scores_df = pd.DataFrame(scores, columns=["model","rmse","mae","r2"]).sort_values("rmse")
print("\n=== CV Summary (lower RMSE is better) ===")
print(scores_df)

best_name = scores_df.iloc[0]["model"]
best_pipe = candidates[best_name]
print(f"\nSelected model: {best_name}")

# ---- refit best on ALL data & save artifacts ----
best_pipe.fit(X, y)
joblib.dump({
    "pipeline": best_pipe,
    "feature_cols": feature_cols,
    "index": X.index,
    "meta": {
        "target": "target_excess_ret_next",
        "option": "A_inner_join_nextQ",
        "n_splits": n_splits
    }
}, OUT/"best_model.joblib")

# ---- full in-sample fitted vs actual & last-available forecast ----
fitted = pd.Series(best_pipe.predict(X), index=y.index, name="y_hat")
fit_df = pd.DataFrame({"y_true": y, "y_hat": fitted})
fit_df.to_csv(OUT/"fitted_vs_actual.csv")

# if you want a “true” next-quarter forecast using the most recent row:
latest_x = X.iloc[[-1]]
latest_quarter = latest_x.index[0]
nextq_pred = float(best_pipe.predict(latest_x))
print(f"\nLatest available quarter = {latest_quarter.date()}  →  predicted next-Q excess_ret = {nextq_pred:.4f}")
with open(OUT/"latest_forecast.txt","w") as f:
    f.write(f"{latest_quarter.date()},{nextq_pred:.6f}\n")

# ---- convenience: export the final training frame ----
data.to_csv(OUT/"model_dataset.csv", index=True)  # index = quarter_end
print(f"\nArtifacts written to: {OUT.resolve()}")


# --- baselines ---
n_splits = 5 if len(X) >= 40 else max(3, len(X)//10)

def eval_cv(estimator, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float)
    rows = []
    for k,(tr,te) in enumerate(tscv.split(X),1):
        Xtr,Xte,ytr,yte = X.iloc[tr],X.iloc[te],y.iloc[tr],y.iloc[te]
        estimator.fit(Xtr,ytr)
        p = pd.Series(estimator.predict(Xte), index=yte.index)
        preds.loc[yte.index] = p
        rows.append({
            "fold":k,
            "start":Xte.index.min().date(),
            "end":Xte.index.max().date(),
            "rmse":mean_squared_error(yte,p,squared=False),
            "mae": mean_absolute_error(yte,p),
            "r2":  r2_score(yte,p)
        })
    # ---- FIX: drop rows that were never in any test set
    valid = preds.notna()
    summary = {
        "rmse": mean_squared_error(y[valid], preds[valid], squared=False),
        "mae":  mean_absolute_error(y[valid], preds[valid]),
        "r2":   r2_score(y[valid], preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds

def baseline_mean(X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float)
    rows = []
    for k,(tr,te) in enumerate(tscv.split(X),1):
        ytr,yte = y.iloc[tr], y.iloc[te]
        mu = ytr.mean()
        p = pd.Series(mu, index=yte.index)
        preds.loc[yte.index] = p
        rows.append({
            "fold":k,
            "start":X.index[te].min().date(),
            "end":X.index[te].max().date(),
            "rmse":mean_squared_error(yte,p,squared=False),
            "mae": mean_absolute_error(yte,p),
            "r2":  r2_score(yte,p)
        })
    # ---- FIX: drop rows that were never in any test set
    valid = preds.notna()
    summary = {
        "rmse": mean_squared_error(y[valid], preds[valid], squared=False),
        "mae":  mean_absolute_error(y[valid], preds[valid]),
        "r2":   r2_score(y[valid], preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds


Data coverage: 2014-03-31 → 2024-09-30 | rows: 43
             excess_ret   cpi_yoy    gdp_yoy  repo  repo_chg_bps  rain_anom
quarter_end                                                                
2024-03-31     0.013611  5.014307   6.366486   6.5           0.0  -5.906736
2024-06-30     0.083945  4.904469   7.384530   6.5           0.0  -5.906736
2024-09-30     0.004268  4.244829  15.613499   6.5           0.0   7.564767
Final features: ['cpi_yoy', 'gdp_yoy', 'repo_chg_bps', 'rain_anom', 'q_sin', 'q_cos', 'excess_ret_lag1', 'excess_ret_ma4']
Rows after cleanup: 39
[ridge] RMSE= 0.146 | MAE= 0.105 | R2=-5.829 | MAPE=445.97%
[lasso] RMSE= 0.083 | MAE= 0.066 | R2=-1.194 | MAPE=172.55%
[gbr] RMSE= 0.087 | MAE= 0.074 | R2=-1.426 | MAPE=243.22%
[rf] RMSE= 0.079 | MAE= 0.065 | R2=-0.975 | MAPE=183.77%

=== CV Summary (lower RMSE is better) ===
   model      rmse       mae        r2
3     rf  0.078677  0.065177 -0.974921
1  lasso  0.082922  0.066241 -1.193767
2    gbr  0.087206  0.074197 

In [3]:
# --- Make sure output folder exists
OUT.mkdir(parents=True, exist_ok=True)

# --- Build CV benchmarks (models + baselines) and save tidy tables
def eval_cv(estimator, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float); rows=[]
    for k,(tr,te) in enumerate(tscv.split(X),1):
        Xtr,Xte,ytr,yte = X.iloc[tr],X.iloc[te],y.iloc[tr],y.iloc[te]
        estimator.fit(Xtr,ytr)
        p = pd.Series(estimator.predict(Xte), index=yte.index)
        preds.loc[yte.index] = p
        rows.append({"fold":k,
                     "start":Xte.index.min().date(),"end":Xte.index.max().date(),
                     "rmse":mean_squared_error(yte,p,squared=False),
                     "mae":mean_absolute_error(yte,p),
                     "r2":r2_score(yte,p)})
    valid = preds.notna()
    summary = {
        "rmse":mean_squared_error(y[valid],preds[valid],squared=False),
        "mae":mean_absolute_error(y[valid],preds[valid]),
        "r2":r2_score(y[valid],preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds

def baseline_mean(X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    preds = pd.Series(index=y.index, dtype=float); rows=[]
    for k,(tr,te) in enumerate(tscv.split(X),1):
        ytr,yte = y.iloc[tr], y.iloc[te]
        mu = ytr.mean()
        p = pd.Series(mu, index=yte.index)
        preds.loc[yte.index] = p
        rows.append({"fold":k,
                     "start":X.index[te].min().date(),"end":X.index[te].max().date(),
                     "rmse":mean_squared_error(yte,p,squared=False),
                     "mae":mean_absolute_error(yte,p),
                     "r2":r2_score(yte,p)})
    valid = preds.notna()
    summary = {
        "rmse":mean_squared_error(y[valid],preds[valid],squared=False),
        "mae":mean_absolute_error(y[valid],preds[valid]),
        "r2":r2_score(y[valid],preds[valid]),
    }
    return pd.DataFrame(rows), pd.Series(summary), preds

n_splits = 5 if len(X) >= 40 else max(3, len(X)//10)

# Baselines
bmean_folds, bmean_sum, bmean_pred = baseline_mean(X, y, n_splits)
blag_folds, blag_sum, blag_pred   = eval_cv(LinearRegression(), X[["excess_ret_lag1"]], y, n_splits)

# Reuse your 'candidates' dict from above
rows, summary = [], []
preds = {"baseline_mean": bmean_pred, "baseline_lag1": blag_pred}

for name, est in candidates.items():
    f, s, p = eval_cv(est, X, y, n_splits)
    f.insert(0,"model",name); rows.append(f)
    summary.append({"model":name, **s.to_dict()})
    preds[name] = p

cv_folds_df = pd.concat(rows, ignore_index=True)
cv_summary_df = pd.concat([
    pd.DataFrame(summary),
    pd.DataFrame([{"model":"baseline_mean", **bmean_sum.to_dict()}]),
    pd.DataFrame([{"model":"baseline_lag1", **blag_sum.to_dict()}]),
], ignore_index=True).sort_values("rmse")

pd.DataFrame({"y_true":y, **preds}).to_csv(OUT/"cv_predictions_w_baselines.csv")
cv_summary_df.to_csv(OUT/"cv_summary_models_vs_baselines.csv", index=False)
cv_folds_df.to_csv(OUT/"cv_folds_metrics_detailed.csv", index=False)

print("Saved:")
print(" -", OUT/"cv_summary_models_vs_baselines.csv")
print(" -", OUT/"cv_folds_metrics_detailed.csv")
print(" -", OUT/"cv_predictions_w_baselines.csv")


Saved:
 - models\cv_summary_models_vs_baselines.csv
 - models\cv_folds_metrics_detailed.csv
 - models\cv_predictions_w_baselines.csv


Hold-out permutation importance (interpretability)

In [4]:
from sklearn.inspection import permutation_importance

best_row = cv_summary_df[~cv_summary_df["model"].isin(["baseline_mean","baseline_lag1"])].iloc[0]
best_name = best_row["model"]
best_est  = candidates[best_name]

h = 6 if len(X) > 10 else max(2, len(X)//5)  # ~1.5 years as holdout
X_tr, y_tr = X.iloc[:-h], y.iloc[:-h]
X_te, y_te = X.iloc[-h:],  y.iloc[-h:]

best_est.fit(X_tr, y_tr)
perm = permutation_importance(best_est, X_te, y_te,
                              n_repeats=200, random_state=42,
                              scoring="neg_root_mean_squared_error")
imp_df = (pd.DataFrame({"feature":X.columns, "importance":perm.importances_mean})
            .sort_values("importance", ascending=False))
imp_df.to_csv(OUT/"permutation_importance_holdout.csv", index=False)
imp_df.head(10)

,feature,importance
6,excess_ret_lag1,0.000849
5,q_cos,0.000187
2,repo_chg_bps,-0.000117
1,gdp_yoy,-0.000139
4,q_sin,-0.000188
7,excess_ret_ma4,-0.000496
3,rain_anom,-0.000611
0,cpi_yoy,-0.002042


Narrative: if excess_ret_lag1 dominates and macros are small/unstable → “momentum matters; macro adds little stable lift.”

EDA correlation matrix

In [5]:
corr = pd.concat([X, y.rename("target_next")], axis=1).corr(numeric_only=True)
corr.to_csv(OUT/"corr_matrix_for_slide.csv")
corr.round(2)

,cpi_yoy,gdp_yoy,repo_chg_bps,rain_anom,q_sin,q_cos,excess_ret_lag1,excess_ret_ma4,target_next
cpi_yoy,1.00,0.25,0.33,-0.19,0.01,-0.02,0.15,0.17,0.23
gdp_yoy,0.25,1.00,0.30,0.18,-0.02,-0.03,0.37,0.44,0.08
repo_chg_bps,0.33,0.30,1.00,0.14,-0.16,0.01,-0.07,0.09,-0.07
rain_anom,-0.19,0.18,0.14,1.00,-0.02,0.00,0.23,0.22,0.04
q_sin,0.01,-0.02,-0.16,-0.02,1.00,-0.00,0.09,0.04,-0.01
q_cos,-0.02,-0.03,0.01,0.00,-0.00,1.00,0.16,0.02,-0.18
excess_ret_lag1,0.15,0.37,-0.07,0.23,0.09,0.16,1.00,0.53,0.01
excess_ret_ma4,0.17,0.44,0.09,0.22,0.04,0.02,0.53,1.00,0.21
target_next,0.23,0.08,-0.07,0.04,-0.01,-0.18,0.01,0.21,1.00


Simple ablation (Momentum-only vs Momentum+Macro)

In [6]:
def run_cv_rmse(est, X, y, n_splits):
    _, s, _ = eval_cv(est, X, y, n_splits)
    return s["rmse"]

mom_cols   = ["excess_ret_lag1","excess_ret_ma4","q_sin","q_cos"]
macro_cols = [c for c in feature_cols if c not in mom_cols]

est = Pipeline([("scaler",StandardScaler()),
                ("model",RandomForestRegressor(random_state=42,n_estimators=500,min_samples_leaf=2))])

rmse_mom   = run_cv_rmse(est, X[mom_cols], y, n_splits)
rmse_full  = run_cv_rmse(est, X[mom_cols + macro_cols], y, n_splits)

ablation = pd.DataFrame({
    "spec":["Momentum only","Momentum + Macro"],
    "rmse":[rmse_mom, rmse_full]
})
ablation.to_csv(OUT/"ablation_momentum_vs_macro.csv", index=False)
ablation


,spec,rmse
0,Momentum only,0.081112
1,Momentum + Macro,0.078753


In [7]:
with pd.ExcelWriter(OUT/"results_for_slides.xlsx") as xw:
    cv_summary_df.to_excel(xw, "cv_summary", index=False)
    cv_folds_df.to_excel(xw, "cv_folds_detailed", index=False)
    imp_df.to_excel(xw, "perm_importance_holdout", index=False)
    corr.to_excel(xw, "corr_matrix")
    ablation.to_excel(xw, "ablation", index=False)
print("Wrote", OUT/"results_for_slides.xlsx")

Wrote models\results_for_slides.xlsx
